In [1]:
print("Hello")

Hello


In [2]:
!jupyter kernelspec list


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Available kernels:
  julia      /root/.local/share/jupyter/kernels/julia
  ir         /usr/local/share/jupyter/kernels/ir
  python3    /usr/local/share/jupyter/kernels/python3


In [3]:
!pip install torch

In [4]:
!pip install -U -q transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00:00:0100:01


In [6]:
from huggingface_hub import notebook_login
notebook_login()

Model

In [72]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"
config_8bit = BitsAndBytesConfig(load_in_8bit=True)
model_8bit = AutoModelForCausalLM.from_pretrained(model_name, 
                                                  quantization_config=config_8bit,
                                                  device_map="auto",
                                                #   torch_dtype=torch.float16,
                                                  trust_remote_code=True,
                                                  torch_dtype=torch.bfloat16
                                                  )

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Tokenizer

In [73]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", trust_remote_code="True")

Dataset

In [74]:
from datasets import load_dataset
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [75]:
dataset["train"][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [76]:
from datasets import DatasetDict

train_test_split = dataset['train'].train_test_split(test_size=0.2
                                                     )
dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': train_test_split['test']
})
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

In [77]:
def to_chat_template(example):

    messages = [
        {"role": "system", "content": example['instruction']},
        {"role": "user", "content": example['input']},
        {"role": "assistant", "content": example['output']}
    ]

    
    return {'text':messages}

dataset = dataset.map(to_chat_template)
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [78]:
dataset['train'][0]

{'input': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.',
 'output': 'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"',
 'instruction': 'translate this sentence in docker command',
 'text': [{'content': 'translate this sentence in docker command',
   'role': 'system'},
  {'content': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.',
   'role': 'user'},
  {'content': 'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"',
   'role': 'assistant'}]}

In [79]:
tokenizer.mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
tokenizer.mistral.apply_chat_template(dataset['train']['text'][0], tokenize=False)

'<s> [INST] translate this sentence in docker command\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image. [/INST] docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"</s>'

In [80]:
tokenizer.apply_chat_template(dataset['train']['text'][0],tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

In [81]:
def apply_chat_template(example):

    new_text = tokenizer.apply_chat_template(example['text'],tokenize=False)
    return {'text': new_text}

dataset = dataset.map(apply_chat_template)
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [82]:
dataset['train']['text'][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

In [83]:
tokenizer(dataset['train']['text'][0])

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 717, 5186, 220, 2366, 21, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 10086, 279, 12827, 11, 4877, 11, 323, 3110, 315, 279, 5448, 430, 1051, 3549, 1603, 279, 5652, 71582, 2217, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 482, 69, 330, 15145, 28, 74661, 25, 19911, 1, 1198, 2293, 48319, 13, 4727, 39254, 3052, 13, 5786, 39254, 3052, 13, 926, 24275, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [84]:
def tokenize_fn(example): 
    return tokenizer(example['text'])

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['input', 'output', 'instruction', 'text'])
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [85]:
len(tokenized_dataset['train']['input_ids'][1])

79

In [86]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors = "pt")

In [87]:
tokenizer.pad_token

In [88]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

#Data Loader

In [89]:

from torch.utils.data import DataLoader

data_loader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 2,
    collate_fn = data_collator
)

for step, batch in enumerate(data_loader):
    print(f"Batch {step}")
    print("input_ids.shape: ", batch['input_ids'].shape)

    if step >= 3: 
        break

Batch 0
input_ids.shape:  torch.Size([2, 88])
Batch 1
input_ids.shape:  torch.Size([2, 79])
Batch 2
input_ids.shape:  torch.Size([2, 80])
Batch 3
input_ids.shape:  torch.Size([2, 78])


LoRa

In [90]:
from peft import LoraConfig, get_peft_model
import copy

model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(

    r = 32,
    lora_alpha = 16, #the higher value will be , lora matrix will get influence than matrix of base model
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_dropout = 0.05,
    bias = "none", 
    task_type = "CAUSAL_LM"  
)

model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [91]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

Hyper Traning

In [ ]:

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
                                    output_dir = "./results",
                                    per_device_train_batch_size = 2,
                                    per_device_eval_batch_size = 2,
                                    eval_steps = 10,
                                    eval_strategy = "steps",
                                    save_steps = 10,
                                    save_strategy = "steps",
                                    num_train_epochs = 1
                                    max_steps = 10,
                                    logging_steps = 10,
                                    learning_rate = 3e-5,
                                    weight_decay = 0.01,
                                    warmup_steps = 0,
                                    fp16 = True,
                                    bf16 = False,
                                    gradient_accumulation_steps = 4,
                                    optim = "paged_adamw_8bit",
                                    report_to = "none"
)

Training

In [99]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [101]:

trainer = SFTTrainer(
    model = model_8bit_lora,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
    # tokenizer = tokenizer,
    args = training_args,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'